In [1]:
import os
import zipfile
import urllib.request

# 1. Direct download URL from Microsoft Open Data
url = "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"
zip_path = "cats_and_dogs.zip"

# 2. Download the dataset
if not os.path.exists(zip_path):
    print("Downloading the dataset (approx. 780 MB)... This will take a moment.")
    urllib.request.urlretrieve(url, zip_path)
    print("Download Complete!")
else:
    print("Zip file already exists.")

# 3. Extract the contents
if not os.path.exists("PetImages"):
    print("Extracting files...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Extraction Complete! 'PetImages' folder is ready.")
else:
    print("Dataset already extracted.")

Download Complete!
Extracting files...
Extraction Complete! 'PetImages' folder is ready.


In [3]:
!pip install opencv-python


   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/40.2 MB ? eta -:--:--
   ---- ----------------------------------- 4.2/40.2 MB 18.0 MB/s eta 0:00:03
   ---------- ----------------------------- 10.5/40.2 MB 23.3 MB/s eta 0:00:02
   --------------- ------------------------ 15.7/40.2 MB 24.1 MB/s eta 0:00:02
   --------------------- ------------------ 21.2/40.2 MB 24.8 MB/s eta 0:00:01
   -------------------------- ------------- 27.0/40.2 MB 25.5 MB/s eta 0:00:01
   -------------------------------- ------- 33.0/40.2 MB 26.2 MB/s eta 0:00:01
   -------------------------------------- - 38.8/40.2 MB 26.5 MB/s eta 0:00:01
   ---------------------------------------- 40.2/40.2 MB 25.3 MB/s  0:00:01



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import cv2
import numpy as np
import random

DATADIR = "PetImages"
CATEGORIES = ["Cat", "Dog"]
IMG_SIZE = 64  # Resizing dimensions to optimize computer memory

data = []

print("Preprocessing images...")
for category in CATEGORIES:
    path = os.path.join(DATADIR, category)
    class_num = CATEGORIES.index(category) # 0 = Cat, 1 = Dog
    
    count = 0
    for img in os.listdir(path):
        if count >= 1200: # Limit to 1200 samples per class to speed up training
            break
        try:
            # Read image in grayscale to collapse color dimensions
            img_array = cv2.imread(os.path.join(path, img), cv2.IMREAD_GRAYSCALE)
            resized_array = cv2.resize(img_array, (IMG_SIZE, IMG_SIZE))
            
            # Flatten 64x64 grid into a single line vector of 4,096 numbers
            flattened = resized_array.flatten()
            
            data.append([flattened, class_num])
            count += 1
        except Exception as e:
            pass # Skip empty or corrupted system files smoothly

# Shuffle data so the model doesn't see all cats first, then all dogs
random.shuffle(data)

# Split features (X) and labels (y)
X = []
y = []
for features, label in data:
    X.append(features)
    y.append(label)

X = np.array(X) / 255.0  # Normalize pixel intensity values between 0 and 1
y = np.array(y)

print(f"Data loading complete! Processed matrix features shape: {X.shape}")

Preprocessing images...
Data loading complete! Processed matrix features shape: (2400, 4096)


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

# Split data into training and validation sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print("Training the Optimized SVM model with RBF Kernel... (This might take a minute)")
# Initialize the SVM classifier with an RBF kernel for non-linear image feature spaces
svm_model = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
svm_model.fit(X_train, y_train)
print("Model training complete!")

# Generate test predictions
y_pred = svm_model.predict(X_test)
print(f"\nModel Accuracy Score: {accuracy_score(y_test, y_pred) * 100:.2f}%")

Training the Optimized SVM model with RBF Kernel... (This might take a minute)
Model training complete!

Model Accuracy Score: 60.62%


In [7]:
print("--- Detailed Classification Report ---")
print(classification_report(y_test, y_pred, target_names=["Cat", "Dog"]))


--- Detailed Classification Report ---
              precision    recall  f1-score   support

         Cat       0.62      0.63      0.62       249
         Dog       0.59      0.58      0.59       231

    accuracy                           0.61       480
   macro avg       0.61      0.61      0.61       480
weighted avg       0.61      0.61      0.61       480

